# NGÀY 1 & NGÀY 2: KIỂM TRA GPU T4, ĐỐI CHIẾU CLEAN_V0 & XÂY DỰNG PYTORCH MULTIMODAL DATASET (43,982 MẪU - 4 LỚP)


In [1]:
import os, glob, time, json, random
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import DistilBertTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Tự động tìm clean_v0.csv, split_ids.csv, label_map.json và thư mục ảnh
CSV_PATH   = glob.glob('/kaggle/input/**/clean_v0*.csv', recursive=True)[0]
SPLIT_PATH = glob.glob('/kaggle/input/**/split_ids*.csv', recursive=True)[0]
LABEL_PATH = glob.glob('/kaggle/input/**/label_map*.json', recursive=True)[0]

df = pd.read_csv(CSV_PATH)
sp = pd.read_csv(SPLIT_PATH)
with open(LABEL_PATH, 'r', encoding='utf-8') as f:
    label_map = json.load(f)

first_img = str(df['image_file'].iloc[0])
IMG_DIR = os.path.dirname(glob.glob(f'/kaggle/input/**/{first_img}', recursive=True)[0])

LABEL_CLASSES = [k for k, v in sorted(label_map.items(), key=lambda x: x[1])]
NUM_CLASSES   = len(LABEL_CLASSES)
MAX_TEXT_LEN  = 48
IMG_SIZE      = 224

print(f"CSV_PATH   : {CSV_PATH} ({len(df)} dòng)")
print(f"SPLIT_PATH : {SPLIT_PATH} ({len(sp)} dòng)")
print(f"IMG_DIR    : {IMG_DIR}")
print(f"LABEL_MAP  : {label_map} (NUM_CLASSES = {NUM_CLASSES})")


Device: cuda | GPU: Tesla T4
CSV_PATH   : /kaggle/input/datasets/nttan28/myntra-cleaned-metadata/clean_v0.csv (43982 dòng)
SPLIT_PATH : /kaggle/input/datasets/nttan28/myntra-cleaned-metadata/split_ids.csv (43982 dòng)
IMG_DIR    : /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/fashion-dataset/images
LABEL_MAP  : {'Casual': 0, 'Ethnic': 1, 'Formal': 2, 'Sports': 3} (NUM_CLASSES = 4)


## T04 & T05: Kiểm tra Schema, Phân phối Split/Label và Tiền xử lý Tabular (Chỉ fit trên Train)


In [2]:
print("Các cột trong clean_v0.csv:", list(df.columns))
print("\nPhân phối tập Train / Val / Test:")
print(df['split'].value_counts())
print("\nPhân phối 4 lớp nhãn (Usage):")
print(df['label'].value_counts())

df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'val'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)

# Tính Class Weights chống mất cân bằng lớp (Casual chiếm đa số)
counts = df_train['label'].value_counts().reindex(LABEL_CLASSES).fillna(1)
raw_w  = (counts.sum() / (NUM_CLASSES * counts)).values
class_weights = torch.tensor(np.sqrt(raw_w), dtype=torch.float32)
print("\nClass Weights (smoothed):", dict(zip(LABEL_CLASSES, class_weights.round(decimals=3).tolist())))

# Tabular Preprocessor: 6 đặc trưng chuẩn Data Contract (gender, masterCategory, subCategory, baseColour, season, year)
TAB_CAT_COLS = ['gender', 'masterCategory', 'subCategory', 'baseColour', 'season']

class TabularPrep:
    def fit(self, d):
        self.maps = {}
        for c in TAB_CAT_COLS:
            vals = sorted(d[c].fillna('Unknown').astype(str).unique())
            self.maps[c] = {v: (i + 1) / max(len(vals), 1) for i, v in enumerate(vals)}
        y = pd.to_numeric(d['year'], errors='coerce').fillna(2012.0)
        self.y_min, self.y_max = float(y.min()), float(y.max())
        self.dim = len(TAB_CAT_COLS) + 1
        return self

    def transform_row(self, row):
        feats = [self.maps[c].get(str(row[c]), 0.0) for c in TAB_CAT_COLS]
        yr = float(row['year']) if pd.notna(row['year']) else 2012.0
        yr_norm = float(np.clip((yr - self.y_min) / (self.y_max - self.y_min + 1e-9), 0.0, 1.0))
        feats.append(yr_norm)
        return feats

tab_prep = TabularPrep().fit(df_train)
print(f"✅ TabularPrep sẵn sàng | TABULAR_DIM = {tab_prep.dim}")


Các cột trong clean_v0.csv: ['id', 'image_file', 'text', 'text_masked', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'label', 'label_idx', 'split']

Phân phối tập Train / Val / Test:
split
train    30787
test      6598
val       6597
Name: count, dtype: int64

Phân phối 4 lớp nhãn (Usage):
label
Casual    34404
Sports     4025
Ethnic     3208
Formal     2345
Name: count, dtype: int64

Class Weights (smoothed): {'Casual': 0.5649999976158142, 'Ethnic': 1.8509999513626099, 'Formal': 2.1659998893737793, 'Sports': 1.652999997138977}
✅ TabularPrep sẵn sàng | TABULAR_DIM = 6


## T06: Xây dựng PyTorch `FashionMultimodalDataset` & Kiểm tra DataLoader 1 Batch + Throughput


In [3]:
IMNET_MEAN, IMNET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class FashionMultimodalDataset(Dataset):
    def __init__(self, d, img_dir, tokenizer, tab_prep, split='train'):
        self.d = d.reset_index(drop=True)
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.tab_prep = tab_prep
        self.broken = 0

        if split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(IMNET_MEAN, IMNET_STD)
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(IMNET_MEAN, IMNET_STD)
            ])

    def __len__(self):
        return len(self.d)

    def __getitem__(self, idx):
        row = self.d.iloc[idx]
        img_path = os.path.join(self.img_dir, str(row['image_file']))
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
            self.broken += 1
        img = self.transform(img)

        txt = str(row['text_masked']) if pd.notna(row['text_masked']) else str(row['text'])
        enc = self.tokenizer(txt, max_length=MAX_TEXT_LEN, padding='max_length', truncation=True, return_tensors='pt')
        tab = torch.tensor(self.tab_prep.transform_row(row), dtype=torch.float32)
        lbl = torch.tensor(int(row['label_idx']), dtype=torch.long)

        return img, enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0), tab, lbl

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
ds_sample = FashionMultimodalDataset(df_train.head(512), IMG_DIR, tokenizer, tab_prep, split='train')
loader = DataLoader(ds_sample, batch_size=32, shuffle=True, num_workers=2)

imgs, input_ids, attn_mask, tabular, labels = next(iter(loader))
print("=" * 60)
print("✅ T06 – DataLoader PASSED (43,982 dataset)")
print(f"  Image shape    : {tuple(imgs.shape)}      # (32, 3, 224, 224)")
print(f"  Input IDs      : {tuple(input_ids.shape)}        # (32, {MAX_TEXT_LEN})")
print(f"  Attention mask : {tuple(attn_mask.shape)}")
print(f"  Tabular        : {tuple(tabular.shape)}         # (32, 6)")
print(f"  Labels         : {labels[:10].tolist()} ...")
print(f"  NaN check      : {torch.isnan(tabular).any().item()}")
print("=" * 60)

t0, n = time.time(), 0
for x, *_ in loader:
    n += x.size(0)
dt = time.time() - t0
print(f"🚀 Throughput: {n/dt:.1f} mẫu/giây ({n} mẫu / {dt:.1f}s)")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✅ T06 – DataLoader PASSED (43,982 dataset)
  Image shape    : (32, 3, 224, 224)      # (32, 3, 224, 224)
  Input IDs      : (32, 48)        # (32, 48)
  Attention mask : (32, 48)
  Tabular        : (32, 6)         # (32, 6)
  Labels         : [0, 0, 0, 3, 0, 0, 0, 3, 0, 0] ...
  NaN check      : False
🚀 Throughput: 17.1 mẫu/giây (512 mẫu / 29.9s)
